# IMDB, Amazon review and Yelp classification with Spacy, TF-IDF and Linear SVC

## 1. Importing Datasets

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [3]:
data_yelp = pd.read_csv('yelp_labelled.txt',sep='\t',header=None) # load restarant data, yelp is a compnay colletcts restaurants review

In [4]:
data_yelp.head()
# review and sentiment
# 0-Negative, 1-Positive for positive review

,0,1
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
# Assign column names
column_name = ['Review','Sentiment']
data_yelp.columns= column_name

In [6]:
data_yelp.head()

,Review,Sentiment
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [7]:
data_yelp.shape
# 1000 rows (reviews), 2 columns (Sentiments)

(1000, 2)

In [8]:
data_amazon = pd.read_csv('amazon_cells_labelled.txt',sep='\t',header=None) # load Amazon review data

In [9]:
data_amazon.head()
# review and sentiment
# 0-Negative, 1-Positive for positive review


,0,1
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [10]:
data_amazon.columns = column_name

In [11]:
data_amazon.head()

,Review,Sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [12]:
data_amazon.shape

(1000, 2)

In [13]:
data_imdb = pd.read_csv('imdb_labelled.txt',sep='\t',header=None) # Load IMDB movie review data

In [14]:
data_imdb.head()

,0,1
0,"A very, very, very slow-moving, aimless movie ...",0
1,Not sure who was more lost - the flat characte...,0
2,Attempting artiness with black & white and cle...,0
3,Very little music or anything to speak of.,0
4,The best scene in the movie was when Gerardo i...,1


In [15]:
data_imdb.columns = column_name

In [16]:
data_imdb.head()

,Review,Sentiment
0,"A very, very, very slow-moving, aimless movie ...",0
1,Not sure who was more lost - the flat characte...,0
2,Attempting artiness with black & white and cle...,0
3,Very little music or anything to speak of.,0
4,The best scene in the movie was when Gerardo i...,1


In [17]:
data_imdb.shape

(748, 2)

In [18]:
# Append all the data in a single dataframe

In [19]:
#data = data_yelp.append([data_amazon, data_imdb],ignore_index=True)
data = pd.concat([data_yelp,data_amazon,data_imdb],ignore_index=True)

In [20]:
data.head()

,Review,Sentiment
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [21]:
data.shape

(2748, 2)

In [22]:
# check distribution of sentiments

In [23]:
data['Sentiment'].value_counts()
# 1386 positive reviews
# 1362 Negative reviews

Sentiment
1    1386
0    1362
Name: count, dtype: int64

In [24]:
# check for null values
data.isnull().sum()

Review       0
Sentiment    0
dtype: int64

In [25]:
x = data['Review']
y = data['Sentiment']

## 2. Data Cleaning

In [26]:
# Here we will remove stopwords, punctuations
# as well as we will apply lemmatization

### Create a funtion to clean the data

In [27]:
import string

In [28]:
punct = string.punctuation

In [29]:
punct

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [30]:
from spacy.lang.en.stop_words import STOP_WORDS

In [31]:
stopwords = list(STOP_WORDS)

In [32]:
print(stopwords)

['everything', "'s", 'often', 'whose', 'give', 'must', 'these', 'elsewhere', 'will', 'former', 'nor', 'latter', 'say', 'made', 'they', 'the', "'d", "'re", 'though', 'just', 'whom', 'move', '‘d', 'is', 'sometimes', 'well', '‘re', 'that', 'amount', 'everywhere', 'my', 'somehow', 'now', 'are', 'might', 'anywhere', 'anyhow', 'besides', 'serious', 'front', 'whither', 'four', 'becomes', 'everyone', 'mostly', 'no', 'never', 'not', "'m", 'with', 'below', 'top', 'nevertheless', 'within', 'whereupon', 'seeming', 'we', 'as', 'yours', 'neither', 'unless', 'ourselves', 'fifteen', 'after', 'side', 'them', 'under', 'almost', 'latterly', 'why', '’d', 'hence', 'ten', 'down', 'please', 'beforehand', 'about', 'anything', 'by', 'namely', 'off', 'of', 'there', 'used', 'anyone', 'cannot', 'themselves', "'ve", 'become', 'eight', 'being', 'most', 'twelve', 'you', 'when', 'again', 'around', 'seemed', 'him', 'wherever', 'mine', 'twenty', 'had', 'any', 'were', 'do', 'such', 'back', 'already', 'keep', 'rather', '

In [33]:
# Creating a function for data cleaning
def text_data_cleaning(sentence):
    doc = nlp(sentence)
    tokens = [] # tokens list

    for token in doc:
        if token.lemma_ != '-PRON-':
            temp = token.lemma_.lower().strip()
        else:
            temp = token.lower_
        tokens.append(temp)
    cleaned_tokens = []
    for token in tokens:
        if token not in stopwords and token not in punct:
            cleaned_tokens.append(token)
    return cleaned_tokens

In [34]:
# if root form of that word is not pronoun then it is going to convert that into lower form
# and if that word is a proper noun, then we are directly taking lower form, because there is no lemma for proper noun

In [35]:
text_data_cleaning("Hello all, It's a beautiful day outside there!")
# stopwords and punctuations removed

['hello', 'beautiful', 'day', 'outside']

### Vectorization feature engineering (TF-IDF)

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

In [37]:
tfidf = TfidfVectorizer(tokenizer=text_data_cleaning)
# tokenizer=text_data_cleaning, tokenization will be done according to this function

In [38]:
classifier = LinearSVC()

## 3. Train the model

### Split the data into train and test dataset

In [39]:
from sklearn.model_selection import train_test_split

In [42]:
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=0)

In [43]:
x_train.shape, x_test.shape

((2198,), (550,))

In [44]:
x_train.head()

2572    An Italian reviewer called this "a small, grea...
526                          And it was way to expensive.
1509    As an earlier review noted, plug in this charg...
144     Nice blanket of moz over top but i feel like t...
2483    The film gives meaning to the phrase, "Never i...
Name: Review, dtype: object

### Fit the x_train and y_train

In [45]:
clf = Pipeline([('tfidf',tfidf),('clf',classifier)])

In [46]:
clf.fit(x_train,y_train)

C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(tokenizer=<function text_data_cleaning at 0x0000024806E514E0>)),
                ('clf', LinearSVC())])

## 4. Predict the test set results 

In [47]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [48]:
y_pred = clf.predict(x_test)

In [49]:
accuracy_score(y_test,y_pred) * 100

76.18181818181819

In [50]:
confusion_matrix(y_test, y_pred)

array([[198,  81],
       [ 50, 221]])

In [51]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.80      0.71      0.75       279
           1       0.73      0.82      0.77       271

    accuracy                           0.76       550
   macro avg       0.77      0.76      0.76       550
weighted avg       0.77      0.76      0.76       550



In [55]:
accuracy_score(y_test, y_pred)
# 76% accuracy

0.7618181818181818

In [56]:
clf.predict(["Wow, I am learning Natural Language Processing in fun fashion!"])
# output is 1, that means review is positive

array([1])

In [57]:
clf.predict(["It's hard to learn new things!"])
# output is 0, that means review is Negative

array([0])